In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging

import os
import sys
import pandas as pd
import numpy as np
import tensorflow as tf # type: ignore

from meridian.model import model
from meridian.model import spec
from meridian.analysis import optimizer
from meridian.analysis import analyzer

from meridian.planner.flex_budget_planner import FlexibleBudgetPlanner, CompareOptimizedVsNonOptimized
from meridian.analysis.optimizer import OptimizationResults


In [17]:
# Configuration for input excel file
input_config = {

  # time and geo inputs
  'time_col': 'week',
  'geo_col': 'geo',
  'population_col': 'population',

  # kpi inputs
  'kpi_col': 'conversions',  #
  'kpi_type': 'non_revenue',
  'revenue_per_kpi_col': 'revenue_per_conversion',  # needed if kpi_type is non_revenue

  # impression based media inputs
  'media_cols': ['Channel0_impression', 'Channel1_impression', 'Channel2_impression'],
  'media_spend_cols': ['Channel0_spend', 'Channel1_spend', 'Channel2_spend'],
  'media_channels': ['Channel0', 'Channel1', 'Channel2'],

}

# optimization config
optimization_config = {
  'fixed_budget': True,
  'use_kpi': True,

  'start_date': '2025-01-04',
  'end_date': '2025-03-29'

}


In [18]:
# Optimizer input excel file path
home_dir = '/Users/mariappan.subramanian/Library/CloudStorage/OneDrive-TheTradeDesk/MMM/BudgetOptimizer'
input_file_path = f'{home_dir}/input_files/lmmm_simulated_geo_data_roi.xlsx'
if not os.path.exists(input_file_path):
  raise FileNotFoundError(f'File not found: {input_file_path}')

In [19]:
# call the planner
planner = FlexibleBudgetPlanner(file_name=input_file_path, model_config=input_config)
opt_results = planner.optimize(optimization_config)

In [20]:
# summarize optimized vs non-optimized
compare_opt_vs_nonopt = CompareOptimizedVsNonOptimized(opt_results)
total_opt_vs_nonopt_df = compare_opt_vs_nonopt.get_total_level_comparison()
channel_opt_vs_nonopt_df = compare_opt_vs_nonopt.get_channel_level_comparison()

In [21]:
total_opt_vs_nonopt_df

,start_date,end_date,optimized_budget,optimized_total_incremental_outcome,optimized_total_cpa,nonoptimized_budget,nonoptimized_total_incremental_outcome,nonoptimized_total_cpa,budget_change,outcome_change,cpa_change
0,2025-01-04,2025-03-29,157.0,903.700684,0.17373,158.0,556.227417,0.284056,-0.006329,0.624696,-0.388396


In [22]:
channel_opt_vs_nonopt_df

,channel,optimized_spend,optimized_incremental_outcome,optimized_effectiveness,optimized_cpa,nonoptimized_spend,nonoptimized_incremental_outcome,nonoptimized_effectiveness,nonoptimized_cpa,budget_change,outcome_change,effectiveness_change,cpa_change
0,Channel0,37,176.339111,0.476592,0.209823,53,221.497314,0.417919,0.239281,-0.301887,-0.203877,0.140392,-0.123109
1,Channel1,52,218.145386,0.419510,0.238373,53,224.352097,0.423306,0.236236,-0.018868,-0.027665,-0.008966,0.009047
2,Channel2,68,509.216217,0.748847,0.133539,52,110.377983,0.212265,0.471108,0.307692,3.613386,2.527883,-0.716544


In [23]:
planner.roi_df

,geo,Channel0,Channel1,Channel2
0,geo0,3.931742,4.032623,2.381835
1,geo1,4.198822,4.301728,2.222216


In [24]:
planner.coefficients_df

,geo,Channel0,Channel1,Channel2
0,geo0,3.737616,7.281049,8.624043
1,geo1,3.991509,7.766928,8.046102


In [3]:
# Excel file path (update this to match the actual location)
excel_file_path = (
  '/Users/mariappan.subramanian/Library/CloudStorage/'
  'OneDrive-TheTradeDesk/MMM/BudgetOptimizer/optimizer_input_case_roi.xlsx'
)

# Model configuration based on actual Excel file structure
model_config = {

  # time and geo inputs
  'time_col': 'week',
  'geo_col': 'geo',
  'population_col': 'population',

  # kpi inputs
  'kpi_type': 'non_revenue',
  'kpi_col': 'conversions',
  'revenue_per_kpi_col': 'revenue_per_conversion',

  # impression based media inputs
  'media_cols': ['Channel0_impression', 'Channel1_impression', 'Channel2_impression'],
  'media_spend_cols': ['Channel0_spend', 'Channel1_spend', 'Channel2_spend'],
  'media_channels': ['Channel0', 'Channel1', 'Channel2'],

  # reach based media inputs
  'reach_cols': ['Channel3_reach'],
  'frequency_cols': ['Channel3_frequency'],
  'rf_spend_cols': ['Channel3_spend'],
  'rf_channels': ['Channel3']

  }

In [ ]:
adl = FlexibleBudgetPlanner(file_name=excel_file_path, model_config=model_config)
adl.load_excel_data()
coeff_df = adl.get_coefficients_data()
roi_data = adl.roi_df  # Access the ROI data directly

In [ ]:
self = FlexibleBudgetPlanner(file_name=excel_file_path, model_config=model_config)

In [6]:
# self.load_excel_data()
self.input_type = self._detect_input_type()
logging.info(f"Detected input type: {self.input_type}")

INFO: Detected input type: roi


In [7]:
# Step 2: Validate sheet requirements for detected type
self._validate_sheet_requirements(self.input_type)

In [8]:
# Step 3: Set is_roi_input flag in model config
self.model_config['is_roi_input'] = (self.input_type == 'roi')

In [9]:
# Step 4: Load Data sheet (always required)
self.data_df = pd.read_excel(self.file_name, sheet_name='Data')
logging.info(f"Loaded Data sheet with shape: {self.data_df.shape}")


INFO: Loaded Data sheet with shape: (3120, 17)


In [10]:
self.data_df.head()

,geo,week,Channel0_impression,Channel1_impression,Channel2_impression,Channel3_impression,competitor_activity_score_control,sentiment_score_control,Channel0_spend,Channel1_spend,Channel2_spend,Channel3_spend,conversions,revenue_per_conversion,population,Channel3_reach,Channel3_frequency
0,Geo0,2021-01-25,1392518,3733,670235,0,-0.783350,3.036792,15482.038,42.681618,8004.1030,0.0000,12530976.0,0.035213,487878.0,0,0.000000
1,Geo0,2021-02-01,937228,722210,745025,226872,-1.834407,-4.244738,10420.116,8257.458000,8897.2630,2668.1526,4926880.5,0.035026,487878.0,181472,1.250176
2,Geo0,2021-02-08,1286569,329778,786262,743321,-1.995511,0.200945,14304.095,3770.548600,9389.7250,8741.9060,10300557.0,0.034529,487878.0,381866,1.946549
3,Geo0,2021-02-15,1149907,529628,190449,400702,-4.925971,-1.542391,12784.685,6055.552700,2274.3865,4712.4990,7398028.5,0.035430,487878.0,273012,1.467708
4,Geo0,2021-02-22,1077028,1057061,584113,346168,-4.541369,0.668494,11974.415,12086.009000,6975.6143,4071.1460,9409540.0,0.034637,487878.0,247506,1.398625


In [11]:
# Step 5: Load Parameters sheet
self.parameters_df = pd.read_excel(self.file_name, sheet_name='Parameters')
logging.info(f"Loaded Parameters sheet with shape: {self.parameters_df.shape}")


INFO: Loaded Parameters sheet with shape: (4, 4)


In [12]:
self.parameters_df

,MediaVariable,Adstock,Inflexion,Slope
0,Channel0,0.510567,1.529526,1.000000
1,Channel1,0.286708,1.232294,1.000000
2,Channel2,0.171263,1.164748,1.000000
3,Channel3,0.478726,1.254190,3.895828


In [13]:
self.roi_df = pd.read_excel(self.file_name, sheet_name='ROI')
logging.info(f"Loaded ROI sheet with shape: {self.roi_df.shape}")
self.roi_df

INFO: Loaded ROI sheet with shape: (20, 5)


,geo,Channel0,Channel1,Channel2,Channel3
0,Geo0,1.643451,2.579856,3.499342,5.129923
1,Geo1,1.678214,2.576494,3.321985,4.913301
2,Geo10,1.674964,2.421661,3.321900,4.909637
3,Geo11,1.634777,2.516862,3.388091,5.153303
4,Geo12,1.712765,2.357297,3.283820,5.204845
5,Geo13,1.571111,2.565322,3.332740,5.241179
6,Geo14,1.496965,2.524445,3.413281,4.950107
7,Geo15,1.601714,2.509541,3.321809,5.058201
8,Geo16,1.765544,2.442872,3.321309,4.879833
9,Geo17,1.637992,2.453314,3.343992,4.979731


In [14]:
# self._convert_roi_to_coefficients()
builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type=self.model_config['kpi_type'],
    default_geo_column=self.model_config['geo_col'],
    default_time_column=self.model_config['time_col'],
    default_population_column=self.model_config['population_col'],
    default_kpi_column=self.model_config['kpi_col']
)

# Add basic data required for ROI conversion
builder = builder.with_kpi(
    self.data_df,
    kpi_col=self.model_config['kpi_col'],
    time_col=self.model_config['time_col'],
    geo_col=self.model_config['geo_col']
)

# Add population data
builder = builder.with_population(
    self.data_df,
    population_col=self.model_config['population_col'],
    geo_col=self.model_config['geo_col']
)

# Add revenue per KPI if available
if 'revenue_per_kpi_col' in self.model_config:
  builder = builder.with_revenue_per_kpi(
      self.data_df,
      revenue_per_kpi_col=self.model_config['revenue_per_kpi_col'],
      time_col=self.model_config['time_col'],
      geo_col=self.model_config['geo_col']
  )

# Add media data
builder = builder.with_media(
    self.data_df,
    media_cols=self.model_config['media_cols'],
    media_spend_cols=self.model_config['media_spend_cols'],
    media_channels=self.model_config['media_channels'],
    time_col=self.model_config['time_col'],
    geo_col=self.model_config['geo_col']
)

# Add RF data if present
if 'rf_channels' in self.model_config and self.model_config['rf_channels']:
  builder = builder.with_reach(
      self.data_df,
      reach_cols=self.model_config['reach_cols'],
      frequency_cols=self.model_config['frequency_cols'],
      rf_spend_cols=self.model_config['rf_spend_cols'],
      rf_channels=self.model_config['rf_channels'],
      time_col=self.model_config['time_col'],
      geo_col=self.model_config['geo_col']
  )

input_data_obj = builder.build()


In [15]:
print(input_data_obj.__class__)
[attr for attr in dir(input_data_obj) if isinstance(getattr(input_data_obj, attr), xr.DataArray)]

<class 'meridian.data.input_data.InputData'>


['allocated_media_spend',
 'allocated_rf_spend',
 'frequency',
 'geo',
 'kpi',
 'media',
 'media_channel',
 'media_spend',
 'media_time',
 'population',
 'reach',
 'revenue_per_kpi',
 'rf_channel',
 'rf_spend',
 'time']

In [16]:
input_data_obj.media.__class__

xarray.core.dataarray.DataArray

In [17]:
# Create converter and perform conversion
self = roi_to_coefficients_converter.ROIToCoefficientsConverter(
    roi_df=self.roi_df,
    parameters_df=self.parameters_df,
    input_data_obj=input_data_obj,
    model_config=self.model_config
)

self.dummy_model = self._create_dummy_model()

INFO: Initialized ROIToCoefficientsConverter with 3 media channels and 1 RF channels


INFO: Successfully created dummy Meridian model for ROI conversion


In [18]:
m = self.dummy_model

# data to be transformed
media_scaled = m.media_tensors.media_scaled

# media parameters
media_params = self._extract_media_parameters()
alpha_m, ec_m, slope_m = media_params['alpha_m'], media_params['ec_m'], media_params['slope_m']

# transformed media
media_transformed = self.dummy_model.adstock_hill_media(
    media=media_scaled,
    alpha=tf.constant(alpha_m, dtype=tf.float32),
    ec=tf.constant(ec_m, dtype=tf.float32),
    slope=tf.constant(slope_m, dtype=tf.float32)
)

In [19]:
# denominator calculation
G, T, M = media_transformed.shape
media_transformed_rev = tf.multiply(media_transformed, m.revenue_per_kpi[:, :, tf.newaxis])
media_transformed_rev_pop = tf.multiply(media_transformed_rev, m.population[:, tf.newaxis, tf.newaxis])
media_contrib_mul = media_transformed_rev_pop * m.kpi_transformer.population_scaled_stdev
media_contrib_mul_gm = tf.reduce_sum(media_contrib_mul, axis=1)

In [20]:
# numerator calc
media_channels = m.input_data.media_channel.values

# step 1: roi tensor
roi_list = []
for chnl in media_channels:
  roi_list.append(self.roi_df[chnl].values)

roi_array = np.array(roi_list).T
roi_tensor = tf.constant(roi_array, dtype=tf.float32)


# step 2: spend tensor
media_spend_gm_xr = m.input_data.media_spend.sum(dim=("time"))
spend_tensor = tf.constant(media_spend_gm_xr, dtype=tf.float32)

# step 3: get the numerator (incremental outcome)
inc_tensor = roi_tensor * spend_tensor

In [21]:
coeff_man = tf.divide(inc_tensor, media_contrib_mul_gm)

In [23]:
coeff_man.numpy()

array([[0.59156954, 0.57695645, 0.6505141 ],
       [0.6054475 , 0.58618826, 0.6153188 ],
       [0.6101154 , 0.5416927 , 0.63429016],
       [0.59452385, 0.5572977 , 0.64696753],
       [0.6213324 , 0.53262335, 0.6198703 ],
       [0.5631951 , 0.5698872 , 0.6427413 ],
       [0.5483471 , 0.5721538 , 0.641096  ],
       [0.5777298 , 0.5744712 , 0.6204898 ],
       [0.63696223, 0.55430424, 0.6330395 ],
       [0.58471215, 0.5413816 , 0.64124703],
       [0.6078178 , 0.55701345, 0.65287364],
       [0.553998  , 0.5128997 , 0.61023784],
       [0.5769996 , 0.5459011 , 0.6181888 ],
       [0.57789373, 0.5547036 , 0.62457067],
       [0.5799759 , 0.54659903, 0.61903095],
       [0.58036155, 0.56963587, 0.6137249 ],
       [0.56195766, 0.5456759 , 0.6337951 ],
       [0.59619737, 0.5530576 , 0.656182  ],
       [0.5734379 , 0.5784693 , 0.6125888 ],
       [0.5837439 , 0.55845106, 0.6381881 ]], dtype=float32)

In [ ]:
adl = FlexibleBudgetPlanner(file_name=excel_file_path, model_config=model_config)
adl.load_excel_data()
coeff_df = adl.get_coefficients_data()
roi_data = adl.roi_df  # Access the ROI data directly

In [30]:
np.allclose(coeff_man.numpy(), coeff_array)

True

In [33]:
np.max(np.abs(coeff_man.numpy() - coeff_array))

np.float32(3.5762787e-07)

In [139]:
converter.parameters_df

,MediaVariable,Adstock,Inflexion,Slope
0,Channel0,0.510567,1.529526,1.000000
1,Channel1,0.286708,1.232294,1.000000
2,Channel2,0.171263,1.164748,1.000000
3,Channel3,0.478726,1.254190,3.895828


In [144]:
adstock_tensors = tf.constant(converter.parameters_df['Adstock'].values, dtype=tf.float32)
ec50_tensors = tf.constant(converter.parameters_df['Inflexion'].values, dtype=tf.float32)
slope_tensors = tf.constant(converter.parameters_df['Slope'].values, dtype=tf.float32)

adstock_tensors[:m.n_media_channels]

<tf.Tensor: shape=(3,), dtype=float32, numpy=array([0.5105672 , 0.28670812, 0.17126298], dtype=float32)>

In [147]:
adstock_tensors[m.n_media_channels:m.n_media_channels + m.n_rf_channels]

<tf.Tensor: shape=(1,), dtype=float32, numpy=array([0.47872618], dtype=float32)>

In [149]:
media_params = converter._extract_media_parameters()
media_params['alpha_m']

[0.5105671882629395, 0.2867081165313721, 0.1712629795074463]

In [150]:
converter._extract_rf_parameters()

{'alpha_rf': [0.4787261784076691],
 'ec_rf': [1.254190444946289],
 'slope_rf': [3.895828247070312]}

In [16]:
input_data_obj

InputData(kpi=<xarray.DataArray 'kpi' (geo: 20, time: 156)> Size: 25kB
array([[12530976. ,  4926880.5, 10300557. , ..., 11532075. ,  9297790. ,
        11967258. ],
       [19017426. , 12329214. , 14298232. , ..., 20809086. , 13530469. ,
         9430961. ],
       [ 4897822.5,  4059605.8,  4581813. , ...,  5128418.5,  4869155. ,
         5191422. ],
       ...,
       [ 2883451.8,  1483594.5,  2006459.6, ...,  1643369.9,  1507497.6,
         1943231. ],
       [16736263. , 12672171. , 11119188. , ..., 16901688. , 21553568. ,
        10977840. ],
       [17849746. , 17027388. , 25654618. , ..., 22121394. , 16636714. ,
        22902830. ]])
Coordinates:
  * time     (time) <U10 6kB '2021-01-25' '2021-02-01' ... '2024-01-15'
  * geo      (geo) <U5 400B 'Geo0' 'Geo1' 'Geo2' ... 'Geo17' 'Geo18' 'Geo19', kpi_type='non_revenue', population=<xarray.DataArray 'population' (geo: 20)> Size: 160B
array([487878.  , 941246.6 , 225414.62, 833550.7 , 157739.06, 598300.25,
       227916.27, 478603.25,

In [24]:
print(input_data_obj.media.shape ) # (G, T, M)
input_data_obj.media

(20, 156, 3)


<xarray.DataArray 'media' (geo: 20, media_time: 156, media_channel: 3)> Size: 75kB
array([[[1392518,    3733,  670235],
        [ 937228,  722210,  745025],
        [1286569,  329778,  786262],
        ...,
        [1211774, 1173873,       0],
        [ 836566,  305098,  407998],
        [1269842, 1263794,  509309]],

       [[3032312, 1231404,  763501],
        [2785805, 1548845, 1290322],
        [2811866, 1705897,  993765],
        ...,
        [2274402, 2511346,   14731],
        [ 786126,       0, 1411209],
        [2067059, 2773346, 1458800]],

       [[ 593030,  438756,  137622],
        [ 534426,  360286,  118274],
        [ 630650,  424657,  326189],
        ...,
...
        ...,
        [ 249370,  264544,   21748],
        [ 183341,   49802,  176608],
        [ 275637,  306384,  106971]],

       [[2582698,  922861,  653117],
        [1424830, 1275632,  962646],
        [2497179, 1203092, 1632313],
        ...,
        [2530296, 1230214,       0],
        [ 946695,  351198, 1537708],
        [2415999, 2758301,  647724]],

       [[2304498,  753260,  255735],
        [2036626, 1433122, 1656609],
        [2946513, 1484241, 1975205],
        ...,
        [2387086, 1703876,       0],
        [ 944313, 1320312, 1500085],
        [2571754, 2509916, 1247641]]])
Coordinates:
  * media_channel  (media_channel) object 24B 'Channel0' 'Channel1' 'Channel2'
  * media_time     (media_time) <U10 6kB '2021-01-25' ... '2024-01-15'
  * geo            (geo) <U5 400B 'Geo0' 'Geo1' 'Geo2' ... 'Geo18' 'Geo19'

1. Create InputData

In [ ]:
flexible_budget_planner = FlexibleBudgetPlanner(file_name=excel_file_path, model_config=model_config)
data = flexible_budget_planner.build_input_data()

In [ ]:
flexible_budget_planner.model_config

------------- Back calculate the coefficients from ROI

In [ ]:
parameter_arrays = flexible_budget_planner.get_processed_parameter_arrays()

In [31]:
dummy_model = model.Meridian(input_data=data, model_spec=spec.ModelSpec())


# get the arguments
mmm = dummy_model
media_tensors = mmm.media_tensors
rf_tensors = mmm.rf_tensors

alpha_m = parameter_arrays.get('alpha_m')
ec_m = parameter_arrays.get('ec_m')
slope_m = parameter_arrays.get('slope_m')

if media_tensors.media is not None:
  media_transformed = mmm.adstock_hill_media(
      media=media_tensors.media_scaled,
      alpha=tf.convert_to_tensor(alpha_m, dtype=tf.float32),
      ec=tf.convert_to_tensor(ec_m, dtype=tf.float32),
      slope=tf.convert_to_tensor(slope_m, dtype=tf.float32),
  )

In [32]:
media_transformed

<tf.Tensor: shape=(20, 156, 3), dtype=float32, numpy=
array([[[0.2908246 , 0.00296332, 0.44369364],
        [0.3267742 , 0.3654256 , 0.50572467],
        [0.38526192, 0.2995557 , 0.526262  ],
        ...,
        [0.41003463, 0.54106367, 0.15925233],
        [0.37518638, 0.3674578 , 0.34121776],
        [0.4047346 , 0.5397538 , 0.40995178]],

       [[0.31641188, 0.33694944, 0.32015955],
        [0.39815843, 0.4397392 , 0.46710268],
        [0.43406695, 0.48160437, 0.43281072],
        ...,
        [0.4123682 , 0.52594423, 0.17299233],
        [0.32331753, 0.24132252, 0.47541517],
        [0.35831326, 0.55271375, 0.5133842 ]],

       [[0.27430627, 0.43054658, 0.26169476],
        [0.3479516 , 0.45581812, 0.2675743 ],
        [0.40277955, 0.49288186, 0.474427  ],
        ...,
        [0.40987837, 0.54559594, 0.38836136],
        [0.35093376, 0.44849288, 0.4184708 ],
        [0.37151918, 0.5361595 , 0.37130007]],

       ...,

       [[0.2974166 , 0.09182433, 0.09301934],
        [0.372

In [33]:
data.media

<xarray.DataArray 'media' (geo: 20, media_time: 156, media_channel: 3)> Size: 75kB
array([[[1392518,    3733,  670235],
        [ 937228,  722210,  745025],
        [1286569,  329778,  786262],
        ...,
        [1211774, 1173873,       0],
        [ 836566,  305098,  407998],
        [1269842, 1263794,  509309]],

       [[3032312, 1231404,  763501],
        [2785805, 1548845, 1290322],
        [2811866, 1705897,  993765],
        ...,
        [2274402, 2511346,   14731],
        [ 786126,       0, 1411209],
        [2067059, 2773346, 1458800]],

       [[ 593030,  438756,  137622],
        [ 534426,  360286,  118274],
        [ 630650,  424657,  326189],
        ...,
...
        ...,
        [ 249370,  264544,   21748],
        [ 183341,   49802,  176608],
        [ 275637,  306384,  106971]],

       [[2582698,  922861,  653117],
        [1424830, 1275632,  962646],
        [2497179, 1203092, 1632313],
        ...,
        [2530296, 1230214,       0],
        [ 946695,  351198, 1537708],
        [2415999, 2758301,  647724]],

       [[2304498,  753260,  255735],
        [2036626, 1433122, 1656609],
        [2946513, 1484241, 1975205],
        ...,
        [2387086, 1703876,       0],
        [ 944313, 1320312, 1500085],
        [2571754, 2509916, 1247641]]])
Coordinates:
  * media_channel  (media_channel) object 24B 'Channel0' 'Channel1' 'Channel2'
  * media_time     (media_time) <U10 6kB '2021-01-25' ... '2024-01-15'
  * geo            (geo) <U5 400B 'Geo0' 'Geo1' 'Geo2' ... 'Geo18' 'Geo19'

In [ ]:
parameter_arrays = flexible_budget_planner.get_processed_parameter_arrays()
coefficient_arrays = flexible_budget_planner.get_processed_coefficients_arrays()

In [6]:
parameter_arrays

{'alpha_m': <xarray.DataArray 'alpha_m' (media_channel: 3)> Size: 24B
 array([0.51056719, 0.28670812, 0.17126298])
 Coordinates:
   * media_channel  (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2',
 'ec_m': <xarray.DataArray 'ec_m' (media_channel: 3)> Size: 24B
 array([1.52952576, 1.23229408, 1.16474795])
 Coordinates:
   * media_channel  (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2',
 'slope_m': <xarray.DataArray 'slope_m' (media_channel: 3)> Size: 24B
 array([1., 1., 1.])
 Coordinates:
   * media_channel  (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2',
 'alpha_rf': <xarray.DataArray 'alpha_rf' (rf_channel: 1)> Size: 8B
 array([0.47872618])
 Coordinates:
   * rf_channel  (rf_channel) <U8 32B 'Channel3',
 'ec_rf': <xarray.DataArray 'ec_rf' (rf_channel: 1)> Size: 8B
 array([1.25419044])
 Coordinates:
   * rf_channel  (rf_channel) <U8 32B 'Channel3',
 'slope_rf': <xarray.DataArray 'slope_rf' (rf_channel: 1)> Size: 8B
 array([3.89582825])
 Coordinates:
   * r

In [8]:
parameter_arrays['alpha_m']

<xarray.DataArray 'alpha_m' (media_channel: 3)> Size: 24B
array([0.51056719, 0.28670812, 0.17126298])
Coordinates:
  * media_channel  (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2'

In [43]:
coefficient_arrays

{'beta_gm': <xarray.DataArray 'beta_gm' (geo: 20, media_channel: 3)> Size: 480B
 array([[0.59156996, 0.57695675, 0.65051496],
        [0.60544699, 0.58618772, 0.61531818],
        [0.5801062 , 0.553298  , 0.6111849 ],
        [0.57807249, 0.55831456, 0.62035668],
        [0.57906914, 0.55886364, 0.62508488],
        [0.57296002, 0.56320864, 0.6191318 ],
        [0.57095027, 0.55212671, 0.6290918 ],
        [0.58986926, 0.56343049, 0.64753032],
        [0.57893503, 0.58381557, 0.62021452],
        [0.5705474 , 0.54913819, 0.63340759],
        [0.61041367, 0.54180527, 0.64330077],
        [0.58996964, 0.56087613, 0.64733016],
        [0.62054336, 0.52024597, 0.63420427],
        [0.5711937 , 0.56435418, 0.64072132],
        [0.54389858, 0.55787253, 0.6380682 ],
        [0.58158261, 0.56385761, 0.63503915],
        [0.63654286, 0.54719704, 0.62848502],
        [0.59715307, 0.55126083, 0.63297832],
        [0.59599978, 0.55974191, 0.6346727 ],
        [0.56065822, 0.51651025, 0.61709756]])

2. Create InferenceData based on the point estimates in the input excel

In [ ]:
inference_data = flexible_budget_planner.get_inference_data()

3. Create MMM object and add InferenceData to it

In [5]:
model_spec = spec.ModelSpec()
mmm = model.Meridian(input_data=data, model_spec=model_spec, inference_data=inference_data)
mmm.sample_prior(n_draws=100, seed=42)  # does not affect the optimization results


In [6]:
mmm.inference_data.posterior

<xarray.Dataset> Size: 5kB
Dimensions:           (chain: 1, draw: 1, media_channel: 3, rf_channel: 1,
                       geo: 20, time: 156, knots: 156, control_variable: 2)
Coordinates:
  * chain             (chain) int64 8B 0
  * draw              (draw) int64 8B 0
  * media_channel     (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2'
  * rf_channel        (rf_channel) <U8 32B 'Channel3'
  * geo               (geo) <U5 400B 'Geo0' 'Geo1' 'Geo10' ... 'Geo8' 'Geo9'
  * time              (time) int64 1kB 0 1 2 3 4 5 6 ... 150 151 152 153 154 155
  * knots             (knots) int64 1kB 0 1 2 3 4 5 ... 150 151 152 153 154 155
  * control_variable  (control_variable) <U33 264B 'sentiment_score_control' ...
Data variables: (12/24)
    alpha_m           (chain, draw, media_channel) float32 12B 0.5106 ... 0.1713
    ec_m              (chain, draw, media_channel) float32 12B 1.53 1.232 1.165
    slope_m           (chain, draw, media_channel) float32 12B 1.0 1.0 1.0
    alpha_rf          (chain, draw, rf_channel) float32 4B 0.4787
    ec_rf             (chain, draw, rf_channel) float32 4B 1.254
    slope_rf          (chain, draw, rf_channel) float32 4B 3.896
    ...                ...
    contribution_rf   (chain, draw, rf_channel) float32 4B 0.0
    beta_rf           (chain, draw, rf_channel) float32 4B 0.0
    eta_rf            (chain, draw, rf_channel) float32 4B 0.0
    gamma_c           (chain, draw, control_variable) float32 8B 0.0 0.0
    xi_c              (chain, draw, control_variable) float32 8B 0.0 0.0
    gamma_gc          (chain, draw, geo, control_variable) float32 160B 0.0 ....

In [38]:
inference_data.posterior

<xarray.Dataset> Size: 5kB
Dimensions:           (chain: 1, draw: 1, media_channel: 3, rf_channel: 1,
                       geo: 20, time: 156, knots: 156, control_variable: 2)
Coordinates:
  * chain             (chain) int64 8B 0
  * draw              (draw) int64 8B 0
  * media_channel     (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2'
  * rf_channel        (rf_channel) <U8 32B 'Channel3'
  * geo               (geo) <U5 400B 'Geo0' 'Geo1' 'Geo10' ... 'Geo8' 'Geo9'
  * time              (time) int64 1kB 0 1 2 3 4 5 6 ... 150 151 152 153 154 155
  * knots             (knots) int64 1kB 0 1 2 3 4 5 ... 150 151 152 153 154 155
  * control_variable  (control_variable) <U33 264B 'sentiment_score_control' ...
Data variables: (12/24)
    alpha_m           (chain, draw, media_channel) float32 12B 0.5106 ... 0.1713
    ec_m              (chain, draw, media_channel) float32 12B 1.53 1.232 1.165
    slope_m           (chain, draw, media_channel) float32 12B 1.0 1.0 1.0
    alpha_rf          (chain, draw, rf_channel) float32 4B 0.4787
    ec_rf             (chain, draw, rf_channel) float32 4B 1.254
    slope_rf          (chain, draw, rf_channel) float32 4B 3.896
    ...                ...
    contribution_rf   (chain, draw, rf_channel) float32 4B 0.0
    beta_rf           (chain, draw, rf_channel) float32 4B 0.0
    eta_rf            (chain, draw, rf_channel) float32 4B 0.0
    gamma_c           (chain, draw, control_variable) float32 8B 0.0 0.0
    xi_c              (chain, draw, control_variable) float32 8B 0.0 0.0
    gamma_gc          (chain, draw, geo, control_variable) float32 160B 0.0 ....

##### ~~~~~~~~ Contribution and ROI calculation ~~~~~~~~~~~~

In [19]:
mmm_analyzer = analyzer.Analyzer(mmm)
inc_outcome = mmm_analyzer.incremental_outcome(use_posterior=True)
inc_outcome.shape

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


2025-09-02 10:21:00.469581: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


TensorShape([1, 1, 4])

In [23]:
inc_outcome

<tf.Tensor: shape=(1, 1, 4), dtype=float32, numpy=array([[[65190904., 68640128., 77200936., 98960488.]]], dtype=float32)>

In [38]:
summary_metrics = mmm_analyzer.summary_metrics(aggregate_geos=False)
geo_summary = summary_metrics.sel(metric='mean',distribution='posterior').to_dataframe()

In [42]:
geo_summary.reset_index().to_csv(os.path.join(tmp_dir, 'geo_roi_summary.csv'))

In [12]:
import tensorflow as tf

# Shapes: g=2, t=5, x=3 (no batch dims)
linear_diff = tf.constant([
  # g = 0  -> 5 rows over t, 3 cols over x
  [[ 1,  2, 3],
    [ 0,  1, 1],
    [-1,  2, 0],
    [ 2,  0, 1],
    [ 1, -1, 2]],
  # g = 1
  [[ 1, -1,  0],
    [ 2,  1,  1],
    [ 0,  2,  1],
    [-1,  0,  2],
    [ 1,  1, -1]],
], dtype=tf.float32)  # shape: (2, 5, 3)

revenue_per_kpi = tf.constant([
  [10, 20, 0, 5, 5],   # g=0
  [ 1,  0, 2, 0, 1],   # g=1
], dtype=tf.float32)      # shape: (2, 5)

population = tf.constant([2, 3], dtype=tf.float32)   # shape: (2,)
population_scaled_stdev = tf.constant(0.5, tf.float32)  # scalar

# --- Einsum version ---
# "...gtx,gt,g,->...gx" with no batch dims ("...") -> "gtx,gt,g,->gx"
out_einsum = tf.einsum("gtx,gt,g,->gx",
                      linear_diff,
                      revenue_per_kpi,
                      population,
                      population_scaled_stdev)

In [13]:
out_einsum

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[25. , 35. , 65. ],
       [ 3. ,  6. ,  1.5]], dtype=float32)>

In [14]:
linear_diff.shape

TensorShape([2, 5, 3])

In [23]:
expression_eval = linear_diff * revenue_per_kpi[:, :, None] * population[:, None, None] * population_scaled_stdev
tf.reduce_sum(expression_eval, axis=-2)

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[25. , 35. , 65. ],
       [ 3. ,  6. ,  1.5]], dtype=float32)>

<tf.Tensor: shape=(2, 1, 1), dtype=float32, numpy=
array([[[2.]],

       [[3.]]], dtype=float32)>

4. Call the Optimizer

<!-- %%time
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize(use_posterior=True) -->

In [10]:
# optimization_results.plot_spend_delta()

In [11]:
# optimization_results.plot_incremental_outcome_delta()

In [12]:
# optimization_results.plot_budget_allocation()

In [13]:
# optimization_results.plot_response_curves()

In [14]:
# optimization_results.optimized_data

3. Let's manually run the optimization routines

In [15]:
from collections.abc import Mapping, Sequence
import dataclasses
import functools
import math
import os
from typing import Any, TypeAlias
import warnings

import altair as alt
import jinja2
from meridian import constants as c
from meridian.analysis import analyzer
from meridian.analysis import formatter
from meridian.analysis import summary_text
from meridian.data import time_coordinates as tc
from meridian.model import model
import numpy as np
import pandas as pd
import tensorflow as tf
import xarray as xr

from meridian.analysis.optimizer import _SpendConstraint, OptimizationGrid, _validate_budget

In [16]:
# initialize
from meridian.analysis import optimizer
self = optimizer.BudgetOptimizer(mmm)

In [17]:
new_data: analyzer.DataTensors | None = None
use_posterior: bool = True
selected_times: tuple[str | None, str | None] | None = None
start_date: tc.Date = None
end_date: tc.Date = None
fixed_budget: bool = True
budget: float | None = None
pct_of_spend: Sequence[float] | None = None
spend_constraint_lower: _SpendConstraint | None = None
spend_constraint_upper: _SpendConstraint | None = None
target_roi: float | None = None
target_mroi: float | None = None
gtol: float = 0.0001
use_optimal_frequency: bool = True
use_kpi: bool = False
confidence_level: float = c.DEFAULT_CONFIDENCE_LEVEL
batch_size: int = c.DEFAULT_BATCH_SIZE
optimization_grid: OptimizationGrid | None = None

In [11]:
if selected_times is not None:
  warnings.warn(
      '`selected_times` is deprecated. Please use `start_date` and'
      ' `end_date` instead.',
      DeprecationWarning,
      stacklevel=2,
  )
  deprecated_start_date, deprecated_end_date = selected_times
  start_date = start_date or deprecated_start_date
  end_date = end_date or deprecated_end_date

_validate_budget(
    fixed_budget=fixed_budget,
    budget=budget,
    target_roi=target_roi,
    target_mroi=target_mroi,
)

In [18]:
spend_constraint_default = (
    c.SPEND_CONSTRAINT_DEFAULT_FIXED_BUDGET
    if fixed_budget
    else c.SPEND_CONSTRAINT_DEFAULT_FLEXIBLE_BUDGET
)

if spend_constraint_lower is None:
  spend_constraint_lower = spend_constraint_default
if spend_constraint_upper is None:
  spend_constraint_upper = spend_constraint_default


In [21]:
spend_constraint_default, spend_constraint_lower, spend_constraint_upper

(0.3, 0.3, 0.3)

In [22]:
use_grid_arg = optimization_grid is not None and self._validate_grid(
    new_data=new_data,
    use_posterior=use_posterior,
    start_date=start_date,
    end_date=end_date,
    budget=budget,
    pct_of_spend=pct_of_spend,
    spend_constraint_lower=spend_constraint_lower,
    spend_constraint_upper=spend_constraint_upper,
    gtol=gtol,
    use_optimal_frequency=use_optimal_frequency,
    use_kpi=use_kpi,
    optimization_grid=optimization_grid,
)

use_grid_arg

False

In [23]:
# if optimization_grid is None or not use_grid_arg:
#   optimization_grid = self.create_optimization_grid(
#       new_data=new_data,
#       start_date=start_date,
#       end_date=end_date,
#       budget=budget,
#       pct_of_spend=pct_of_spend,
#       spend_constraint_lower=spend_constraint_lower,
#       spend_constraint_upper=spend_constraint_upper,
#       gtol=gtol,
#       use_posterior=use_posterior,
#       use_kpi=use_kpi,
#       use_optimal_frequency=use_optimal_frequency,
#       batch_size=batch_size,
#   )

# inside create_optimization_grid
new_data=new_data
start_date=start_date
end_date=end_date
budget=budget
pct_of_spend=pct_of_spend
spend_constraint_lower=spend_constraint_lower
spend_constraint_upper=spend_constraint_upper
gtol=gtol
use_posterior=use_posterior
use_kpi=use_kpi
use_optimal_frequency=use_optimal_frequency
batch_size=batch_size


In [25]:
self._validate_model_fit(use_posterior)
if new_data is None:
  new_data = analyzer.DataTensors()

required_tensors = c.PERFORMANCE_DATA + (c.TIME,)
filled_data = new_data.validate_and_fill_missing_data(
    required_tensors_names=required_tensors, meridian=self._meridian
)

In [38]:
hist_spend = self._analyzer.get_aggregated_spend(
    new_data=filled_data.filter_fields(c.PAID_CHANNELS + c.SPEND_DATA),
    selected_times=selected_times,
    include_media=self._meridian.n_media_channels > 0,
    include_rf=self._meridian.n_rf_channels > 0,
).data

In [41]:
hist_spend

array([40414664., 27601934., 23265924., 19630668.], dtype=float32)

In [42]:
new_data_hist = filled_data.filter_fields(c.PAID_CHANNELS + c.SPEND_DATA)
tf.reduce_sum(new_data_hist.media_spend, axis=(0, 1))

<tf.Tensor: shape=(3,), dtype=float32, numpy=array([40414664., 27601934., 23265924.], dtype=float32)>

In [31]:
c.PAID_CHANNELS + c.SPEND_DATA

('media', 'reach', 'frequency', 'media_spend', 'rf_spend')

In [45]:
mmm.input_data.media_channel.values

array(['Channel0', 'Channel1', 'Channel2'], dtype=object)

In [46]:
media_spend_tensor = filled_data.media_spend
media_spend_tensor.ndim


3

In [47]:
allowed_n_channels = [
    mmm.n_media_channels,
    mmm.n_rf_channels,
    mmm.n_media_channels + mmm.n_rf_channels,
    mmm.n_media_channels
    + mmm.n_rf_channels
    + mmm.n_non_media_channels
    + mmm.n_organic_media_channels
    + mmm.n_organic_rf_channels,
]

In [49]:
has_media_dim = True
aggregate_geos = True
aggregate_times = True
tensor_dims = "...gt" + "m" * has_media_dim
output_dims = (
    "g" * (not aggregate_geos)
    + "t" * (not aggregate_times)
    + "m" * has_media_dim
)

In [50]:
tensor_dims

'...gtm'

In [51]:
output_dims

'm'

In [52]:
tensor = media_spend_tensor
tf.einsum(f"{tensor_dims}->...{output_dims}", tensor)

<tf.Tensor: shape=(3,), dtype=float32, numpy=array([40414664., 27601934., 23265924.], dtype=float32)>

In [54]:
f"{tensor_dims}->...{output_dims}"

'...gtm->...m'

In [53]:
tf.reduce_sum(tensor, axis=(0, 1))

<tf.Tensor: shape=(3,), dtype=float32, numpy=array([40414664., 27601934., 23265924.], dtype=float32)>

In [ ]:
optimization_grid.grid_dataset

<xarray.Dataset> Size: 18kB
Dimensions:                   (grid_spend_index: 243, channel: 4)
Coordinates:
  * grid_spend_index          (grid_spend_index) int64 2kB 0 1 2 ... 240 241 242
  * channel                   (channel) object 32B 'Channel0' ... 'Channel3'
Data variables:
    spend_grid                (grid_spend_index, channel) float64 8kB 2.83e+0...
    incremental_outcome_grid  (grid_spend_index, channel) float64 8kB 5.179e+...
Attributes:
    spend_step_size:  100000

In [18]:
mean_opt_results = optimization_grid.grid_dataset.mean(dim="grid_spend_index")
mean_opt_results.to_pandas()

,spend_grid,incremental_outcome_grid
channel,,
Channel0,40400000.0,6.470639e+07
Channel1,27600000.0,6.812717e+07
Channel2,23300000.0,7.669325e+07
Channel3,19600000.0,1.062761e+08


In [6]:
# from meridian.analysis import optimizer
# budget_optimizer = optimizer.BudgetOptimizer(mmm)
# optimization_results = budget_optimizer.optimize()

In [6]:
# param_arrays_dict = adhoc_data_loader.get_processed_parameter_arrays()
# coeff_arrays_dict = adhoc_data_loader.get_processed_coefficients_arrays()

INFO: Coefficients validation passed - all requirements satisfied
INFO: Successfully created coefficients DataArrays for 2 coefficient types
INFO: Successfully processed media coefficients as DataArrays


In [7]:
# from meridian.model import model
# from meridian.model import spec


# # Configure the model
# model_spec = spec.ModelSpec()
# mmm = model.Meridian(input_data=data, model_spec=model_spec)

In [14]:
from unittest import mock

from meridian.data import input_data

In [15]:
mock_data = mock.MagicMock(input_data)
mock_data

<MagicMock spec='module' id='5820205344'>

<MagicMock name='mock.InputData' id='5820203712'>